In [ ]:
!pip install pandas numpy scikit-learn openpyxl


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report


In [ ]:
file_path = "health_nutrition_disease_dataset_12000 (1).xlsx"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
print("\nColumns in dataset:")
print(df.columns)

df.head()


In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Fill missing numeric values
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

# Encode categorical columns
label_encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("Cleaning complete!")
df.head()


In [ ]:
target_cols = [col for col in df.columns if "Risk" in col]

print("Target columns detected:")
print(target_cols)

X = df.drop(columns=target_cols)
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, random_state=42)
)

print("Training model...")
model.fit(X_train, y_train)
print("Training complete!")


In [ ]:
y_pred = model.predict(X_test)

for i, disease in enumerate(target_cols):
    print(f"\n===== {disease} =====")
    print(classification_report(y_test.iloc[:, i], y_pred[:, i]))
